In [34]:
import re

import numpy as np
import pandas as pd

In [35]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5645 entries, 0 to 5644
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date_start  5645 non-null   object
 1   date_end    603 non-null    object
 2   event       5645 non-null   object
dtypes: object(3)
memory usage: 132.4+ KB


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

ru_stopwords = stopwords.words("russian")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


texts = df["event"].dropna().astype(str).apply(clean_text)

vectorizer = TfidfVectorizer(
    max_df=0.9,
    min_df=5,
    stop_words=ru_stopwords,
    ngram_range=(1, 3),
    sublinear_tf=True,
)
X = vectorizer.fit_transform(texts)

n_topics = 10
model = NMF(n_components=n_topics, random_state=42)
W = model.fit_transform(X)
H = model.components_

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic in enumerate(H):
    top_words = [feature_names[j] for j in topic.argsort()[:-11:-1]]
    topics[f"Topic {i + 1}"] = top_words

print(len(vectorizer.vocabulary_))
topics

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ruslan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


3311


{'Topic 1': ['парламентские выборы',
  'парламентские',
  'выборы',
  'досрочные парламентские',
  'досрочные парламентские выборы',
  'досрочные',
  'партия',
  'большинство',
  'сирии',
  'мест'],
 'Topic 2': ['человек',
  'погибли',
  'погибли человек',
  'результате',
  'человека',
  'человек погибли',
  'получили',
  'ранены',
  'ранения',
  'погибло'],
 'Topic 3': ['должность',
  'вступил',
  'должность президента',
  'президента',
  'вступил должность',
  'вступил должность президента',
  'года',
  'президент',
  'должность президент',
  'вступил должность президент'],
 'Topic 4': ['тур',
  'выборов',
  'президентских',
  'президентских выборов',
  'второй',
  'второй тур',
  'тур президентских',
  'тур президентских выборов',
  'второй тур президентских',
  'одержал'],
 'Topic 5': ['мира',
  'чемпионат',
  'чемпионат мира',
  'россия',
  'мира хоккею',
  'хоккею',
  'чемпионат мира хоккею',
  'сборная',
  'мира хоккею шайбой',
  'хоккею шайбой'],
 'Topic 6': ['премьер',
  'мини

In [38]:
from sklearn.decomposition import TruncatedSVD

n_topics = 10

lsa = TruncatedSVD(
    n_components=n_topics,
    random_state=42
)

X_lsa = lsa.fit_transform(X)
feature_names = vectorizer.get_feature_names_out()

topics = {}

for i, comp in enumerate(lsa.components_):
    indices = np.argsort(np.abs(comp))[-12:]
    top_words = [feature_names[j] for j in indices]
    topics[f"Topic {i + 1}"] = top_words

topics

{'Topic 1': ['тур',
  'победу одержала',
  'победу одержал',
  'одержал',
  'одержала',
  'партия',
  'президентские выборы',
  'президентские',
  'победу',
  'парламентские выборы',
  'парламентские',
  'выборы'],
 'Topic 2': ['президента',
  'погибло',
  'получили ранения',
  'ранены',
  'ранения',
  'получили',
  'человек погибли',
  'человека',
  'результате',
  'погибли человек',
  'погибли',
  'человек'],
 'Topic 3': ['одержал',
  'погибли',
  'человек',
  'тур',
  'выборов',
  'парламентские',
  'парламентские выборы',
  'вступил должность',
  'должность президента',
  'вступил',
  'президента',
  'должность'],
 'Topic 4': ['одержал',
  'победу',
  'второй',
  'президентских выборов',
  'второй тур',
  'президентских',
  'вступил должность',
  'должность президента',
  'выборов',
  'тур',
  'вступил',
  'должность'],
 'Topic 5': ['хоккею шайбой',
  'мира хоккею шайбой',
  'шайбой',
  'одержала',
  'чемпионат мира хоккею',
  'хоккею',
  'мира хоккею',
  'сборная',
  'россия',
  '

In [39]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=30,
    doc_topic_prior=0.1,  # alpha
    topic_word_prior=0.01  # beta
)
X_lda = lda.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic_dist in enumerate(lda.components_):
    top_idx = topic_dist.argsort()[-12:][::-1]
    topics[f"Topic {i + 1}"] = [feature_names[j] for j in top_idx]

topics

{'Topic 1': ['стал',
  'министром',
  'премьер министром',
  'премьер',
  'новым',
  'референдум',
  'сша',
  'президентом',
  'новым премьер',
  'новым премьер министром',
  'великобритании',
  'конституционный'],
 'Topic 2': ['космический',
  'союз',
  'россии',
  'рф',
  'союз тма',
  'тма',
  'корабль',
  'аппарат',
  'посадки',
  'сша',
  'казахстане',
  'представителей'],
 'Topic 3': ['россия',
  'впервые',
  'сша',
  'европы',
  'станции',
  'истории',
  'второй тур выборов',
  'тур выборов',
  'старт',
  'выборов президента',
  'полёт',
  'оон'],
 'Topic 4': ['человек',
  'погибли',
  'результате',
  'погибли человек',
  'человека',
  'погибло',
  'ранены',
  'получили',
  'человек погибли',
  'ранения',
  'взрыв',
  'получили ранения'],
 'Topic 5': ['парламентские выборы',
  'парламентские',
  'выборы',
  'премьер',
  'президент',
  'министра',
  'премьер министра',
  'года',
  'должность',
  'министр',
  'вступил',
  'отставку'],
 'Topic 6': ['сша',
  'корабля',
  'россии',
 

In [40]:
import os
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

texts = df["event"]

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    min_df=5
)

MODEL_PATH = "../models/bertopic_events_auto"

if os.path.exists(MODEL_PATH):
    topic_model = BERTopic.load(MODEL_PATH, embedding_model=embedding_model)
    topics = topic_model.topics_
    print(f"Модель загружена из {MODEL_PATH}")
else:
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        language="multilingual",
        calculate_probabilities=True,
        verbose=True,
    )
    topics, probs = topic_model.fit_transform(texts)
    topic_model.save(MODEL_PATH)
    print(f"Модель обучена и сохранена в {MODEL_PATH}")

Модель загружена из ../models/bertopic_events_auto


In [41]:
import random
import socket
import uuid
from collections import defaultdict

from dash import Dash, html, dcc


def _find_free_port(min_port=10001):
    port = min_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("localhost", port)) != 0:
                return port
        port += 1


def serve_figure(fig, title=""):
    """Запускает Dash-сервер для отображения Plotly-фигуры. Выводит ссылку в консоль."""
    app = Dash(f"fig_{uuid.uuid4().hex[:8]}")
    app.layout = html.Div([dcc.Graph(figure=fig)])
    port = _find_free_port()
    app.run(port=port, jupyter_mode="external")


def sample_docs_per_topic(texts, topics, n_samples=10, seed=42):
    random.seed(seed)

    topic_to_docs = defaultdict(list)
    for text, topic in zip(texts, topics):
        topic_to_docs[topic].append(text)

    for topic_id, docs in sorted(topic_to_docs.items()):
        if topic_id == -1:
            print(f"topic {topic_id} | total docs: {len(docs)}")
            continue

        print("=" * 80)
        print(f"TOPIC {topic_id} | total docs: {len(docs)}")
        print("=" * 80)

        sampled = random.sample(docs, min(n_samples, len(docs)))
        for i, doc in enumerate(sampled, 1):
            print(f"{i}. {doc}")
        print()


def visualize_topics_barchart(topic_model, top_n_topics=30):
    fig = topic_model.visualize_barchart(top_n_topics=top_n_topics, n_words=10)
    serve_figure(fig, "Barchart:")


def visualize_documents_scatter(topic_model, texts, topics):
    fig = topic_model.visualize_documents(docs=texts, topics=topics)
    serve_figure(fig, "Scatter:")

In [42]:
print('=== Auto-модель: scatter документов по темам ===')
visualize_documents_scatter(topic_model, texts, topics)

=== Auto-модель: scatter документов по темам ===
Dash app running on http://127.0.0.1:10011/


In [43]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1527,-1_на_тур_премьер_по,"[на, тур, премьер, по, парламентские выборы, ч...",[второй тур президентских выборов в Иране. Поб...
1,0,188,0_перу_всеобщие_чили_всеобщие выборы,"[перу, всеобщие, чили, всеобщие выборы, презид...",[Второй тур президентских выборов в Уругвае. П...
2,1,139,1_протеста_против_протесты_массовые,"[протеста, против, протесты, массовые, началис...",[в Армении начались протесты против уступок по...
3,2,89,2_результате взрыва_взрыва_результате_погибли,"[результате взрыва, взрыва, результате, погибл...",[В результате взрыва на химическом заводе во ф...
4,3,87,3_сша_президент сша_трампа_дональда трампа,"[сша, президент сша, трампа, дональда трампа, ...",[Коллегия выборщиков утвердила победу Дональда...
5,4,87,4_выборы_всеобщие выборы_всеобщие_президентские,"[выборы, всеобщие выборы, всеобщие, президентс...",[Президентские выборы в Кот-д’Ивуаре. Победу о...
6,5,86,5_самолёт_авиакомпании_борту_на борту,"[самолёт, авиакомпании, борту, на борту, все, ...",[Под Далянем разбился авиалайнер McDonnell Dou...
7,6,80,6_произошло_землетрясения_человек_более,"[произошло, землетрясения, человек, более, без...",[В Тбилиси произошло землетрясение магнитудой ...
8,7,73,7_австрии_германии_парламентские выборы_партия,"[австрии, германии, парламентские выборы, парт...",[Парламентские выборы в Австрии. По предварите...
9,8,71,8_саммит_нато_государств_брикс,"[саммит, нато, государств, брикс, глав, снг, г...","[саммит АТЭС (Манила, Филиппины)., саммит ШОС ..."


In [44]:
sample_docs_per_topic(texts, topics, n_samples=10)
print('=== Auto-модель: barchart тем ===')
visualize_topics_barchart(topic_model)

topic -1 | total docs: 1527
TOPIC 0 | total docs: 188
1. президент Эквадора Даниэль Нобоа ввёл чрезвычайное положение в стране после побега из тюрьмы строгого режима одного из лидеров наркокартеля «Лос Чонерос» и начала вооружённых столкновений между правительством и наркокартелями.
2. в Венесуэле совершена попытка государственного переворота. Свергнут президент Уго Чавес, распущены парламент и Верховный суд. Временным президентом стал Педро Кармона. На следующий день Чавес восстановлен в должности, а Кармона арестован.
3. Альберто Фухимори переизбран президентом Перу. Оппозиция не признала выборы и ответила новыми маршами протеста, в которых участвовали сотни тысяч людей. Полиция жестоко подавляла выступления. В ходе столкновений сотни людей были ранены и арестованы.
4. выборы президента Венесуэлы. Победу одержал Уго Чавес.
5. VI-й съезд Коммунистической партии Кубы принял решение о серьёзных политических и экономических реформах в стране.
6. на очередных президентских выборах в Панам

In [45]:
import os

MODEL_PATH_15 = "../models/bertopic_events_15topics"

if os.path.exists(MODEL_PATH_15):
    topic_model = BERTopic.load(MODEL_PATH_15, embedding_model=embedding_model)
    topics = topic_model.topics_
    print(f"Модель загружена из {MODEL_PATH_15}")
else:
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        language="multilingual",
        calculate_probabilities=True,
        verbose=True,
        zeroshot_topic_list=[
            "природные катаклизмы", "авиакатастрофы", "террористические нападения", "техногенные аварии", "войны",
            "санкции", "выборы", "нефть", "крипт"
        ],
        seed_topic_list=[
            ["землетрясение", "цунами", "извержение вулкана", "тайфун", "ураган",
             "наводнение", "торнадо", "смерч", "засуха", "лесной пожар"],
            ["авиакатастрофа", "крушение самолёта", "столкновение воздушное",
             "на борту", "рейс", "Boeing"],
            ["террористические нападения", "теракт", "взрыв", "боевики", "заложники",
             "смертник", "перестрелка", "ИГИЛ", "талибы вооружённые"],
            ["техногенные аварии", "утечка", "авария атомная", "загрязнение", "пожар", "АЭС", "взрыв на заводе"],
            ["война", "конфликт", "вторжение"],
            ["санкции", "пакет санкций"],
            ["выборы", "голос", "инаугурация", "импичмент"],
            ["рынок нефти", "баррель", "ОПЕК"],
            ["биткоин", "криптовалюта"],
            ["ракетный пуск", "ядерная программа", "ядерное оружие"],
            ["Nvidia", "Microsoft", "Google", "Samsung", "Huawei", "Facebook", "Apple"]
        ]
    )
    topics, probs = topic_model.fit_transform(texts)
    topic_model.reduce_topics(texts, nr_topics=15)
    topics = topic_model.topics_
    topic_model.save(MODEL_PATH_15)
    print(f"Модель обучена и сохранена в {MODEL_PATH_15}")

topic_model.get_topic_info()

Модель загружена из ../models/bertopic_events_15topics


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1370,-1_по_на_мира_россии,"[по, на, мира, россии, президент, президента, ...",[В первом туре парламентских выборов в Маврита...
1,0,1043,0_победу_выборов_премьер_президента,"[победу, выборов, премьер, президента, тур, вт...",[второй тур выборов президента Польши. Победу ...
2,1,782,1_человек_погибли_результате_война,"[человек, погибли, результате, война, на, ране...",[Во Владикавказе на вещевом рынке «Фаллой» взо...
3,2,694,2_союз_на_погибли_человек,"[союз, на, погибли, человек, сша, станции, чел...",[старт космического корабля Союз ТМА-09М к меж...
4,3,455,3_россии_рф_россия_на,"[россии, рф, россия, на, российской, украины, ...",[инаугурация президента России Владимира Влади...
5,4,350,4_мира_по_саммит_европы,"[мира, по, саммит, европы, международный, сост...",[чемпионат мира по спортивной гимнастике (Бель...
6,5,280,5_человек_погибли_более_произошло,"[человек, погибли, более, произошло, без, резу...","[землетрясение магнитудой до 7,1 в Китае. Поги..."
7,6,150,6_военный_президента_президент_ким,"[военный, президента, президент, ким, чен, южн...",[Ким Сок Су назначен на пост премьер-министра ...
8,7,133,7_закон_европейского_ес_совета,"[закон, европейского, ес, совета, силу, стала,...",[Польша стала государством-председателем Совет...
9,8,111,8_отношения_израиль_израиля_между,"[отношения, израиль, израиля, между, соглашени...",[Хорватия и Камерун установили дипломатические...


In [46]:
topics, probs = topic_model.transform(texts)

Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-03-27 23:58:20,636 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


In [47]:
sample_docs_per_topic(texts, topics, n_samples=10)

topic -1 | total docs: 243
TOPIC 0 | total docs: 1163
1. в Молдавии состоялся второй тур всеобщих местных выборов. По всей территории Молдавии процент явки избирателей составил 58,68 %, а в Кишинёве 45,43 %, что достаточно для признания выборов состоявшимися. На выборах были избраны районные и муниципальные, городские и сельские советы, а также 1295 примаров.
2. введение национальной валюты, «сомони», в Республике Таджикистан.
3. Выборы президента Колумбии.
4. президентские выборы на Мадагаскаре.
5. Парламентские выборы в Италии.
6. парламентские выборы в Словакии, формирование коалиции социал-демократической партии Курс — социальная демократия, популистского Движения за демократическую Словакию и националистической Словацкой национальной партии.
7. в Кыргызстане по результатам референдума были одобрены поправки к Конституции, а президент Аскар Акаев получил возможность оставаться в должности до 2005 года.
8. вступила в должность президент Молдовы Майя Санду.
9. Дрис Жетту назначен пре

In [48]:
print('=== 15-topics модель: barchart тем ===')
visualize_topics_barchart(topic_model)

=== 15-topics модель: barchart тем ===
Dash app running on http://127.0.0.1:10013/


In [49]:
print('=== 15-topics модель: scatter документов по темам ===')
visualize_documents_scatter(topic_model, texts, topics)

=== 15-topics модель: scatter документов по темам ===
Dash app running on http://127.0.0.1:10014/
